In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_WORKING_DIRECTORY = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate_directory
        for candidate_directory in (
            NOTEBOOK_WORKING_DIRECTORY,
            *NOTEBOOK_WORKING_DIRECTORY.parents,
        )
        if (
            (candidate_directory / "models" / "metrics").is_dir()
            and (candidate_directory / "reports").is_dir()
            and (candidate_directory / "configs").is_dir()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root containing models/, "
        "reports/, and configs/."
    )

os.chdir(PROJECT_ROOT)

TABLE_DIRECTORY = PROJECT_ROOT / "reports" / "tables"
FIGURE_DIRECTORY = PROJECT_ROOT / "reports" / "figures"

TABLE_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)

CALIBRATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "models"
    / "metrics"
    / "calibration_comparison_validation.json"
)

VALIDATION_POLICY_PATH = (
    PROJECT_ROOT
    / "models"
    / "metrics"
    / "phase5_validation_policy_evaluation.json"
)

HOLDOUT_POLICY_PATH = (
    PROJECT_ROOT
    / "models"
    / "metrics"
    / "phase5_final_holdout_evaluation.json"
)

with CALIBRATION_RESULTS_PATH.open(encoding="utf-8") as file:
    calibration_results = json.load(file)

with VALIDATION_POLICY_PATH.open(encoding="utf-8") as file:
    validation_policy_results = json.load(file)

with HOLDOUT_POLICY_PATH.open(encoding="utf-8") as file:
    holdout_policy_results = json.load(file)

calibration_rows = []

for method_name, method_result in calibration_results["results"].items():
    calibration_rows.append(
        {
            "method": method_name,
            "brier_score": method_result["brier_score"],
            "expected_calibration_error": (
                method_result["expected_calibration_error"]
            ),
            "pr_auc": method_result["pr_auc"],
            "roc_auc": method_result["roc_auc"],
            "reliability_bin_count": len(
                method_result["mean_predicted_probability"]
            ),
        }
    )

calibration_comparison_table = (
    pd.DataFrame(calibration_rows)
    .sort_values(
        ["brier_score", "expected_calibration_error"],
        ascending=True,
    )
    .reset_index(drop=True)
)

selected_calibration_method = calibration_results["selection_rule"][
    "provisional_selected_method"
]

if selected_calibration_method != "sigmoid":
    raise ValueError(
        "Expected the Phase 5 selected calibration method to be sigmoid."
    )

calibration_comparison_table.to_csv(
    TABLE_DIRECTORY / "phase7c_calibration_comparison.csv",
    index=False,
)

reliability_rows = []

for method_name, method_result in calibration_results["results"].items():
    for bin_number, (
        predicted_probability,
        observed_fraud_rate,
        bin_count,
    ) in enumerate(
        zip(
            method_result["mean_predicted_probability"],
            method_result["observed_fraud_rate"],
            method_result["bin_counts"],
            strict=True,
        ),
        start=1,
    ):
        reliability_rows.append(
            {
                "method": method_name,
                "bin_number": bin_number,
                "mean_predicted_probability": predicted_probability,
                "observed_fraud_rate": observed_fraud_rate,
                "bin_count": bin_count,
                "absolute_calibration_gap": abs(
                    predicted_probability - observed_fraud_rate
                ),
            }
        )

reliability_table = pd.DataFrame(reliability_rows)

reliability_table.to_csv(
    TABLE_DIRECTORY / "phase7c_reliability_curve_data.csv",
    index=False,
)

capacity_sensitivity_table = pd.DataFrame(
    validation_policy_results["capacity_sensitivity_results"]
).sort_values("review_capacity", ignore_index=True)

capacity_sensitivity_table.to_csv(
    TABLE_DIRECTORY / "phase7c_capacity_sensitivity.csv",
    index=False,
)

selected_validation_result = validation_policy_results[
    "selected_capacity_result"
]

holdout_result = holdout_policy_results["holdout_result"]

validation_holdout_policy_comparison = pd.DataFrame(
    [
        {
            "evaluation_period": "chronological_validation_policy_selection",
            "policy_version": selected_validation_result["policy_version"],
            "review_capacity": selected_validation_result[
                "review_capacity"
            ],
            "selected_review_count": selected_validation_result[
                "selected_review_count"
            ],
            "total_fraud_count": selected_validation_result[
                "total_fraud_count"
            ],
            "captured_fraud_count": selected_validation_result[
                "captured_fraud_count"
            ],
            "fraud_capture_rate": selected_validation_result[
                "fraud_capture_rate"
            ],
            "false_positive_review_count": selected_validation_result[
                "false_positive_review_count"
            ],
            "total_expected_review_cost": selected_validation_result[
                "total_expected_review_cost"
            ],
            "total_expected_prevention_value": selected_validation_result[
                "total_expected_prevention_value"
            ],
            "net_expected_value": selected_validation_result[
                "net_expected_value"
            ],
        },
        {
            "evaluation_period": "final_chronological_holdout",
            "policy_version": holdout_result["policy_version"],
            "review_capacity": holdout_result["review_capacity"],
            "selected_review_count": holdout_result[
                "selected_review_count"
            ],
            "total_fraud_count": holdout_result["total_fraud_count"],
            "captured_fraud_count": holdout_result[
                "captured_fraud_count"
            ],
            "fraud_capture_rate": holdout_result[
                "fraud_capture_rate"
            ],
            "false_positive_review_count": holdout_result[
                "false_positive_review_count"
            ],
            "total_expected_review_cost": holdout_result[
                "total_expected_review_cost"
            ],
            "total_expected_prevention_value": holdout_result[
                "total_expected_prevention_value"
            ],
            "net_expected_value": holdout_result["net_expected_value"],
        },
    ]
)

validation_holdout_policy_comparison.to_csv(
    TABLE_DIRECTORY / "phase7c_validation_holdout_policy_comparison.csv",
    index=False,
)

calibration_policy_metadata = pd.DataFrame(
    [
        {
            "model_name": calibration_results["champion_model"]["model_name"],
            "model_version": calibration_results["champion_model"][
                "model_version"
            ],
            "mlflow_run_id": calibration_results["champion_model"][
                "mlflow_run_id"
            ],
            "selected_calibration_method": selected_calibration_method,
            "calibration_primary_selection_metric": (
                calibration_results["selection_rule"]["primary_metric"]
            ),
            "calibration_secondary_selection_metric": (
                calibration_results["selection_rule"]["secondary_metric"]
            ),
            "calibration_fit_rows": calibration_results[
                "validation_split"
            ]["calibration_fit_rows"],
            "policy_selection_rows": validation_policy_results[
                "chronological_split"
            ]["policy_selection_rows"],
            "configured_operational_capacity": (
                validation_policy_results[
                    "configured_operational_capacity"
                ]
            ),
            "final_holdout_evaluation_complete": (
                holdout_policy_results["data_protection"][
                    "holdout_evaluation_complete"
                ]
            ),
            "currency": holdout_policy_results[
                "policy_configuration"
            ]["currency"],
        }
    ]
)

calibration_policy_metadata.to_csv(
    TABLE_DIRECTORY / "phase7c_calibration_policy_metadata.csv",
    index=False,
)

plt.figure(figsize=(9, 7))

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.2,
    color="black",
    label="Perfect calibration",
)

method_colours = {
    "uncalibrated": "#6B7280",
    "sigmoid": "#2563EB",
    "isotonic": "#DC2626",
}

for method_name in ["uncalibrated", "sigmoid", "isotonic"]:
    method_curve = reliability_table.loc[
        reliability_table["method"] == method_name
    ]

    plt.plot(
        method_curve["mean_predicted_probability"],
        method_curve["observed_fraud_rate"],
        marker="o",
        linewidth=2,
        color=method_colours[method_name],
        label=method_name.title(),
    )

plt.xlim(0, 0.75)
plt.ylim(0, 0.30)
plt.xlabel("Mean predicted fraud probability")
plt.ylabel("Observed fraud rate")
plt.title(
    "Phase 7C: Reliability Curves on Chronological Validation Selection Data"
)
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIRECTORY / "phase7c_calibration_reliability_curves.png",
    dpi=200,
    bbox_inches="tight",
)
plt.close()

plt.figure(figsize=(11, 5))

metric_plot_table = calibration_comparison_table.set_index("method")

plt.subplot(1, 2, 1)
plt.bar(
    metric_plot_table.index,
    metric_plot_table["brier_score"],
    color=[
        method_colours[method_name]
        for method_name in metric_plot_table.index
    ],
)
plt.ylabel("Brier score")
plt.title("Lower is better")
plt.xticks(rotation=20, ha="right")

plt.subplot(1, 2, 2)
plt.bar(
    metric_plot_table.index,
    metric_plot_table["expected_calibration_error"],
    color=[
        method_colours[method_name]
        for method_name in metric_plot_table.index
    ],
)
plt.ylabel("Expected calibration error")
plt.title("Lower is better")
plt.xticks(rotation=20, ha="right")

plt.suptitle("Phase 7C: Calibration Comparison")
plt.tight_layout()
plt.savefig(
    FIGURE_DIRECTORY / "phase7c_calibration_metric_comparison.png",
    dpi=200,
    bbox_inches="tight",
)
plt.close()

plt.figure(figsize=(11, 6))

plt.plot(
    capacity_sensitivity_table["review_capacity"],
    capacity_sensitivity_table["fraud_capture_rate"],
    marker="o",
    linewidth=2,
    color="#16A34A",
    label="Fraud capture rate",
)

plt.xlabel("Review capacity")
plt.ylabel("Fraud capture rate")
plt.ylim(bottom=0)
plt.title(
    "Phase 7C: Validation Fraud Capture by Review Capacity"
)
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIRECTORY / "phase7c_capacity_capture_curve.png",
    dpi=200,
    bbox_inches="tight",
)
plt.close()

plt.figure(figsize=(11, 6))

plt.plot(
    capacity_sensitivity_table["review_capacity"],
    capacity_sensitivity_table["net_expected_value"],
    marker="o",
    linewidth=2,
    color="#2563EB",
)

plt.xlabel("Review capacity")
plt.ylabel("Illustrative net expected value (GBP)")
plt.title(
    "Phase 7C: Validation Expected Value by Review Capacity"
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIRECTORY / "phase7c_capacity_expected_value_curve.png",
    dpi=200,
    bbox_inches="tight",
)
plt.close()

print("\n=== PHASE 7C CALIBRATION AND POLICY ANALYSIS COMPLETE ===")
print(
    "Selected calibration method: "
    f"{selected_calibration_method}"
)

selected_calibration_row = calibration_comparison_table.loc[
    calibration_comparison_table["method"]
    == selected_calibration_method
].iloc[0]

uncalibrated_row = calibration_comparison_table.loc[
    calibration_comparison_table["method"] == "uncalibrated"
].iloc[0]

print(
    "Brier score: "
    f"{float(uncalibrated_row['brier_score']):.6f} uncalibrated -> "
    f"{float(selected_calibration_row['brier_score']):.6f} sigmoid"
)
print(
    "ECE: "
    f"{float(uncalibrated_row['expected_calibration_error']):.6f} "
    "uncalibrated -> "
    f"{float(selected_calibration_row['expected_calibration_error']):.6f} "
    "sigmoid"
)
print(
    "Selected validation policy at capacity 1,000: "
    f"{int(selected_validation_result['captured_fraud_count']):,} / "
    f"{int(selected_validation_result['total_fraud_count']):,} fraud "
    f"captured; GBP "
    f"{float(selected_validation_result['net_expected_value']):,.2f} "
    "illustrative net expected value"
)
print(
    "Final holdout policy at capacity 1,000: "
    f"{int(holdout_result['captured_fraud_count']):,} / "
    f"{int(holdout_result['total_fraud_count']):,} fraud captured; GBP "
    f"{float(holdout_result['net_expected_value']):,.2f} "
    "illustrative net expected value"
)

print("\nCalibration comparison:")
display(calibration_comparison_table)

print("\nValidation capacity sensitivity:")
display(capacity_sensitivity_table)

print("\nValidation versus holdout policy comparison:")
display(validation_holdout_policy_comparison)

print("\nSaved figures:")
for figure_path in sorted(FIGURE_DIRECTORY.glob("phase7c_*.png")):
    print(figure_path.relative_to(PROJECT_ROOT))

print("\nSaved tables:")
for table_path in sorted(TABLE_DIRECTORY.glob("phase7c_*.csv")):
    print(table_path.relative_to(PROJECT_ROOT))


=== PHASE 7C CALIBRATION AND POLICY ANALYSIS COMPLETE ===
Selected calibration method: sigmoid
Brier score: 0.068533 uncalibrated -> 0.022892 sigmoid
ECE: 0.161188 uncalibrated -> 0.003719 sigmoid
Selected validation policy at capacity 1,000: 568 / 1,449 fraud captured; GBP 279,000.00 illustrative net expected value
Final holdout policy at capacity 1,000: 870 / 3,083 fraud captured; GBP 430,000.00 illustrative net expected value

Calibration comparison:


,method,brier_score,expected_calibration_error,pr_auc,roc_auc,reliability_bin_count
0,sigmoid,0.022892,0.003719,0.458580,0.897459,10
1,isotonic,0.022902,0.004235,0.441714,0.896277,9
2,uncalibrated,0.068533,0.161188,0.458580,0.897459,10



Validation capacity sensitivity:


,policy_version,selected_review_count,review_capacity,capacity_utilisation,captured_fraud_count,total_fraud_count,fraud_capture_rate,false_positive_review_count,total_expected_review_cost,total_expected_prevention_value,net_expected_value
0,threshold-policy-v1.0.0-capacity-500,500,500,1.0,392,1449,0.270531,108,2500.0,196000.0,193500.0
1,threshold-policy-v1.0.0-capacity-1000,1000,1000,1.0,568,1449,0.391994,432,5000.0,284000.0,279000.0
2,threshold-policy-v1.0.0-capacity-1500,1500,1500,1.0,655,1449,0.452036,845,7500.0,327500.0,320000.0
3,threshold-policy-v1.0.0-capacity-2000,2000,2000,1.0,731,1449,0.504486,1269,10000.0,365500.0,355500.0



Validation versus holdout policy comparison:


,evaluation_period,policy_version,review_capacity,selected_review_count,total_fraud_count,captured_fraud_count,fraud_capture_rate,false_positive_review_count,total_expected_review_cost,total_expected_prevention_value,net_expected_value
0,chronological_validation_policy_selection,threshold-policy-v1.0.0-capacity-1000,1000,1000,1449,568,0.391994,432,5000.0,284000.0,279000.0
1,final_chronological_holdout,threshold-policy-v1.0.0-holdout,1000,1000,3083,870,0.282193,130,5000.0,435000.0,430000.0



Saved figures:
reports\figures\phase7c_calibration_metric_comparison.png
reports\figures\phase7c_calibration_reliability_curves.png
reports\figures\phase7c_capacity_capture_curve.png
reports\figures\phase7c_capacity_expected_value_curve.png

Saved tables:
reports\tables\phase7c_calibration_comparison.csv
reports\tables\phase7c_calibration_policy_metadata.csv
reports\tables\phase7c_capacity_sensitivity.csv
reports\tables\phase7c_reliability_curve_data.csv
reports\tables\phase7c_validation_holdout_policy_comparison.csv


In [ ]:
from pathlib import Path

import pandas as pd

REPORT_DIRECTORY = PROJECT_ROOT / "reports" / "evaluation"
REPORT_DIRECTORY.mkdir(parents=True, exist_ok=True)

calibration_comparison = pd.read_csv(
    TABLE_DIRECTORY / "phase7c_calibration_comparison.csv"
)

reliability_curve_data = pd.read_csv(
    TABLE_DIRECTORY / "phase7c_reliability_curve_data.csv"
)

capacity_sensitivity = pd.read_csv(
    TABLE_DIRECTORY / "phase7c_capacity_sensitivity.csv"
)

validation_holdout_comparison = pd.read_csv(
    TABLE_DIRECTORY / "phase7c_validation_holdout_policy_comparison.csv"
)

metadata = pd.read_csv(
    TABLE_DIRECTORY / "phase7c_calibration_policy_metadata.csv"
).iloc[0].to_dict()


def markdown_table(
    table: pd.DataFrame,
    columns: list[str],
    decimal_columns: list[str] | None = None,
) -> str:
    """Create a Markdown table without requiring optional packages."""
    decimal_columns = decimal_columns or []
    display_table = table.loc[:, columns].copy()

    for column in decimal_columns:
        if column in display_table.columns:
            display_table[column] = display_table[column].map(
                lambda value: (
                    f"{float(value):.6f}"
                    if pd.notna(value)
                    else ""
                )
            )

    display_table = display_table.fillna("")

    for column in display_table.columns:
        display_table[column] = (
            display_table[column]
            .astype(str)
            .str.replace("|", "\\|", regex=False)
        )

    header = "| " + " | ".join(display_table.columns) + " |"
    separator = "| " + " | ".join(
        ["---"] * len(display_table.columns)
    ) + " |"

    rows = [
        "| " + " | ".join(row) + " |"
        for row in display_table.astype(str).values.tolist()
    ]

    return "\n".join([header, separator, *rows])


selected_method = "sigmoid"

uncalibrated_row = calibration_comparison.loc[
    calibration_comparison["method"] == "uncalibrated"
].iloc[0]

sigmoid_row = calibration_comparison.loc[
    calibration_comparison["method"] == "sigmoid"
].iloc[0]

isotonic_row = calibration_comparison.loc[
    calibration_comparison["method"] == "isotonic"
].iloc[0]

validation_policy_row = validation_holdout_comparison.loc[
    validation_holdout_comparison["evaluation_period"]
    == "chronological_validation_policy_selection"
].iloc[0]

holdout_policy_row = validation_holdout_comparison.loc[
    validation_holdout_comparison["evaluation_period"]
    == "final_chronological_holdout"
].iloc[0]

capacity_500_row = capacity_sensitivity.loc[
    capacity_sensitivity["review_capacity"] == 500
].iloc[0]

capacity_1000_row = capacity_sensitivity.loc[
    capacity_sensitivity["review_capacity"] == 1000
].iloc[0]

capacity_2000_row = capacity_sensitivity.loc[
    capacity_sensitivity["review_capacity"] == 2000
].iloc[0]

top_uncalibrated_bin = (
    reliability_curve_data.loc[
        reliability_curve_data["method"] == "uncalibrated"
    ]
    .sort_values("mean_predicted_probability")
    .iloc[-1]
)

top_sigmoid_bin = (
    reliability_curve_data.loc[
        reliability_curve_data["method"] == "sigmoid"
    ]
    .sort_values("mean_predicted_probability")
    .iloc[-1]
)

report_content = f"""# Phase 7C — Calibration and Policy Report

## Purpose

This report compares uncalibrated and calibrated fraud probabilities, documents
the validation-based calibration selection process, evaluates the constrained
review-capacity policy, and summarises policy sensitivity at different review
capacities.

All results are from the public IEEE-CIS Fraud Detection benchmark under
documented illustrative cost assumptions. Expected-value figures are simulated
policy results and must not be described as real financial-institution savings.

## Frozen Scope

| Item | Value |
| --- | --- |
| Model name | `{metadata["model_name"]}` |
| Model version | `{metadata["model_version"]}` |
| MLflow training run | `{metadata["mlflow_run_id"]}` |
| Selected calibration method | `{metadata["selected_calibration_method"]}` |
| Primary calibration-selection metric | `{metadata["calibration_primary_selection_metric"]}` |
| Secondary calibration-selection metric | `{metadata["calibration_secondary_selection_metric"]}` |
| Calibration-fit rows | `{int(metadata["calibration_fit_rows"]):,}` |
| Policy-selection rows | `{int(metadata["policy_selection_rows"]):,}` |
| Locked operational review capacity | `{int(metadata["configured_operational_capacity"]):,}` |
| Currency for illustrative assumptions | `{metadata["currency"]}` |
| Final holdout evaluation complete | `{metadata["final_holdout_evaluation_complete"]}` |

Calibration selection used chronological validation data only. The earliest
validation period fit the calibrators, and the later validation period selected
the calibration method and review policy. The final chronological holdout was
kept separate until the final evaluation.

## Calibration Comparison

The primary selection criterion was Brier score, with expected calibration error
(ECE) used as the secondary criterion. Lower values are better for both metrics.

![Calibration metric comparison](../figures/phase7c_calibration_metric_comparison.png)

{markdown_table(
    calibration_comparison,
    columns=[
        "method",
        "brier_score",
        "expected_calibration_error",
        "pr_auc",
        "roc_auc",
        "reliability_bin_count",
    ],
    decimal_columns=[
        "brier_score",
        "expected_calibration_error",
        "pr_auc",
        "roc_auc",
    ],
)}

Sigmoid / Platt scaling was selected because it achieved the lowest Brier score
of `{float(sigmoid_row["brier_score"]):.6f}` and the lowest ECE of
`{float(sigmoid_row["expected_calibration_error"]):.6f}`. Compared with the
uncalibrated model, Brier score improved from
`{float(uncalibrated_row["brier_score"]):.6f}` to
`{float(sigmoid_row["brier_score"]):.6f}`, while ECE improved from
`{float(uncalibrated_row["expected_calibration_error"]):.6f}` to
`{float(sigmoid_row["expected_calibration_error"]):.6f}`.

Isotonic calibration was close on calibration metrics, with Brier score
`{float(isotonic_row["brier_score"]):.6f}` and ECE
`{float(isotonic_row["expected_calibration_error"]):.6f}`, but did not outperform
sigmoid calibration under the documented selection rule.

The PR-AUC for uncalibrated and sigmoid probabilities remained
`{float(sigmoid_row["pr_auc"]):.6f}` because monotonic sigmoid calibration
preserves ranking. Isotonic calibration produced PR-AUC
`{float(isotonic_row["pr_auc"]):.6f}` in the saved validation comparison.

## Reliability Evidence

![Reliability curves](../figures/phase7c_calibration_reliability_curves.png)

The uncalibrated model was substantially overconfident in the highest recorded
reliability bin: mean predicted probability was
`{float(top_uncalibrated_bin["mean_predicted_probability"]):.6f}`, while observed
fraud rate was `{float(top_uncalibrated_bin["observed_fraud_rate"]):.6f}`.

For the sigmoid-calibrated output, the highest recorded bin had mean predicted
probability `{float(top_sigmoid_bin["mean_predicted_probability"]):.6f}` and
observed fraud rate `{float(top_sigmoid_bin["observed_fraud_rate"]):.6f}`.
This closer alignment supports the selection of sigmoid calibration for
probability-based decision interpretation.

## Capacity-Constrained Policy

The final policy is capacity constrained: transactions are prioritised by model
risk ranking, and the highest-ranked cases are selected until the configured
review capacity is reached. This is preferable to presenting a standalone fixed
probability threshold as an operational recommendation, because a threshold alone
does not guarantee that investigation demand stays within available capacity.

At the selected validation capacity of
`{int(capacity_1000_row["review_capacity"]):,}`, the policy reviewed
`{int(capacity_1000_row["selected_review_count"]):,}` transactions, captured
`{int(capacity_1000_row["captured_fraud_count"]):,}` of
`{int(capacity_1000_row["total_fraud_count"]):,}` fraud cases, and produced an
illustrative net expected value of {metadata["currency"]}
`{float(capacity_1000_row["net_expected_value"]):,.2f}`.

## Capacity Sensitivity

![Capacity capture curve](../figures/phase7c_capacity_capture_curve.png)

![Capacity expected value curve](../figures/phase7c_capacity_expected_value_curve.png)

{markdown_table(
    capacity_sensitivity,
    columns=[
        "review_capacity",
        "selected_review_count",
        "captured_fraud_count",
        "total_fraud_count",
        "fraud_capture_rate",
        "false_positive_review_count",
        "total_expected_review_cost",
        "total_expected_prevention_value",
        "net_expected_value",
    ],
    decimal_columns=[
        "fraud_capture_rate",
        "total_expected_review_cost",
        "total_expected_prevention_value",
        "net_expected_value",
    ],
)}

Increasing validation capacity from
`{int(capacity_500_row["review_capacity"]):,}` to
`{int(capacity_2000_row["review_capacity"]):,}` increased fraud capture rate from
`{float(capacity_500_row["fraud_capture_rate"]):.6f}` to
`{float(capacity_2000_row["fraud_capture_rate"]):.6f}`. It also increased
legitimate review burden from
`{int(capacity_500_row["false_positive_review_count"]):,}` to
`{int(capacity_2000_row["false_positive_review_count"]):,}`.

The configured capacity of 1,000 is therefore a documented operational
assumption, not an arbitrary score cutoff. Phase 7D will frame these capacity
trade-offs in the final business-value summary.

## Validation and Holdout Results

{markdown_table(
    validation_holdout_comparison,
    columns=[
        "evaluation_period",
        "review_capacity",
        "selected_review_count",
        "captured_fraud_count",
        "total_fraud_count",
        "fraud_capture_rate",
        "false_positive_review_count",
        "total_expected_review_cost",
        "total_expected_prevention_value",
        "net_expected_value",
    ],
    decimal_columns=[
        "fraud_capture_rate",
        "total_expected_review_cost",
        "total_expected_prevention_value",
        "net_expected_value",
    ],
)}

On the validation policy-selection period, the 1,000-case review cohort captured
`{int(validation_policy_row["captured_fraud_count"]):,}` of
`{int(validation_policy_row["total_fraud_count"]):,}` fraud cases
(`{float(validation_policy_row["fraud_capture_rate"]):.6f}` capture rate).

On the untouched final chronological holdout, the same 1,000-case capacity
captured `{int(holdout_policy_row["captured_fraud_count"]):,}` of
`{int(holdout_policy_row["total_fraud_count"]):,}` fraud cases
(`{float(holdout_policy_row["fraud_capture_rate"]):.6f}` capture rate).

The absolute illustrative expected-value figures are not directly comparable
between validation and holdout because the evaluation periods have different
numbers of transactions and fraud cases. Capture rate, review precision, review
volume, and cost assumptions provide the more meaningful capacity-normalised
comparison.

## Limitations

- Calibration quality was evaluated on chronological validation periods, but no
  calibration approach guarantees identical probability calibration in every
  future time period.
- Sigmoid calibration preserves ranking, which is useful for top-k review
  prioritisation, but calibration can still affect probability-based thresholds
  and cost estimates.
- The review capacity of 1,000 is a documented benchmark assumption rather than
  a real operational staffing limit.
- A capacity-constrained top-k cohort is the primary operational comparison.
  A threshold without a capacity control can create unpredictable review volume.
- Expected value uses illustrative GBP cost assumptions and should not be
  interpreted as realised savings, prevented losses, or an institutional forecast.
- Final-holdout results are one chronological benchmark evaluation and should not
  be used for further model, threshold, or calibrator selection.

## Output Inventory

```text
reports/figures/phase7c_calibration_metric_comparison.png
reports/figures/phase7c_calibration_reliability_curves.png
reports/figures/phase7c_capacity_capture_curve.png
reports/figures/phase7c_capacity_expected_value_curve.png
reports/tables/phase7c_calibration_comparison.csv
reports/tables/phase7c_calibration_policy_metadata.csv
reports/tables/phase7c_capacity_sensitivity.csv
reports/tables/phase7c_reliability_curve_data.csv
reports/tables/phase7c_validation_holdout_policy_comparison.csv
```
"""

report_path = REPORT_DIRECTORY / "calibration_and_policy_report.md"
report_path.write_text(report_content, encoding="utf-8")

print("=== PHASE 7C REPORT CREATED ===")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Report size: {report_path.stat().st_size:,} bytes")
print(
    "Selected calibration: "
    f"{selected_method}"
)
print(
    "Sigmoid Brier score: "
    f"{float(sigmoid_row['brier_score']):.6f}"
)
print(
    "Final holdout capture rate at capacity 1,000: "
    f"{float(holdout_policy_row['fraud_capture_rate']):.6f}"
)